# Limpeza, Tratamento e Sanitização dos Dados de Alarmes PSD (2023-2026)

## 1. Contexto e Objetivos do Pipeline de Limpeza

O objetivo deste notebook é executar o tratamento rigoroso das bases históricas de alarmes das Portas de Plataforma (PSDs). O dataset bruto é composto pela unificação de dois arquivos disponibilizados (`Alarmes do sistema de PSD.xlsx` e `Histórico de alarmes 23-26.xlsx`), os quais contêm registros operacionais, marcas temporais e identificadores técnicos de equipamentos.

Para garantir a confiabilidade de modelos preditivos e das análises exploratórias, este pipeline executa as seguintes etapas:
1. **Unificação e Desduplicação Rigorosa:** Consolidação dos datasets e eliminação de registros exatamente duplicados.
2. **Tratamento de Valores Nulos e Ausentes:** Análise e imputação ou remoção criteriosa de registros incompletos.
3. **Conversão e Padronização de Tipos de Dados:** Conversão de strings temporais em segundos numéricos e timestamps padrão (`datetime64[ns]`).
4. **Decomposição Estruturada de TAGs (Regex Robusto):** Parsing preciso da coluna `TAG` tratando os dois padrões de nomenclatura (`PSD_EST_EST1_PDM09_TIPO` e `PSD_P_EST1_MDU16_TIPO`) para garantir a correta extração das estações metroviárias reais (ex: `BUT`, `PIN`, `LUZ`, `FRA`).
5. **Detecção e Tratamento de Anomalias Temporais e Outliers:** Identificação de incoerências de marcas de tempo (ex: tempo de normalização negativo ou duração superior a limites operacionais plausíveis).
6. **Visualizações Gráficas do Pipeline:** Gráficos demonstrando o afunilamento de volume e a distribuição corrigida por estação.
7. **Exportação do Dataset Higienizado:** Salvamento da base tratada em formato CSV.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.family'] = 'sans-serif'
plt.rcParams['axes.edgecolor'] = '#cccccc'
plt.rcParams['axes.linewidth'] = 0.8
plt.rcParams['figure.dpi'] = 120

path_base1 = "../../data/Alarmes/Alarmes do sistema de PSD.xlsx"
path_base2 = "../../data/Alarmes/Histórico de alarmes 23-26.xlsx"

print("Carregando Base 1...")
df1 = pd.read_excel(path_base1)
print(f"Base 1 carregada: {df1.shape[0]:,} linhas e {df1.shape[1]} colunas.")

print("Carregando Base 2...")
df2 = pd.read_excel(path_base2)
print(f"Base 2 carregada: {df2.shape[0]:,} linhas e {df2.shape[1]} colunas.")



## 2. Unificação, Eliminação de Duplicados e Visualização de Afunilamento

Ao integrar diferentes extrações históricas do sistema de supervisão de alarmes, é comum encontrar registros duplicados decorrentes de sobreposição de janelas de exportação.

Nesta etapa, concatenamos as duas tabelas e aplicamos a desduplicação de registros idênticos, acompanhada por um gráfico de barras demonstrando o volume mantido versus o descartado.


In [ ]:
df_raw = pd.concat([df1, df2], ignore_index=True)
total_inicial = len(df_raw)

qtd_duplicados = df_raw.duplicated().sum()
df_dedup = df_raw.drop_duplicates().copy()

print(f"Total de registros unificados (bruto): {total_inicial:,}")
print(f"Registros exatamente duplicados identificados: {qtd_duplicados:,} ({qtd_duplicados/total_inicial*100:.2f}%)")
print(f"Total de registros únicos mantidos: {len(df_dedup):,}")

fig, ax = plt.subplots(figsize=(8, 4.5))
categorias = ['Registros Únicos', 'Registros Duplicados']
valores = [len(df_dedup), qtd_duplicados]
cores = ['#4c78a8', '#e15759']

bars = ax.bar(categorias, valores, color=cores, width=0.5, edgecolor='black', linewidth=0.8)

for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 15000, f"{yval:,} ({(yval/total_inicial)*100:.1f}%)", ha='center', va='bottom', fontweight='bold')

ax.set_title('Proporção de Registros Únicos vs. Duplicados na Base Bruta', fontsize=12, fontweight='bold', pad=15)
ax.set_ylabel('Quantidade de Registros', fontsize=10, fontweight='bold')
ax.set_ylim(0, total_inicial * 1.1)
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()



## 3. Diagnóstico e Tratamento de Valores Ausentes (Missing Values)

Examinamos a presença de valores nulos em cada coluna para definir a estratégia de higienização:
* Registros sem **TAG** ou **Horário do Alarme** impedem a localização física e temporal do evento, devendo ser removidos.
* Registros com pequenos nulos na indicação de estação ou porta podem ser inferidos diretamente através da decompilação da string da **TAG**.


In [ ]:
nulos_df = pd.DataFrame({
    'Quantidade_Nulos': df_dedup.isnull().sum(),
    'Percentual (%)': (df_dedup.isnull().sum() / len(df_dedup)) * 100
})
display(nulos_df)

fig, ax = plt.subplots(figsize=(10, 4))
nulos_filtrados = nulos_df[nulos_df['Quantidade_Nulos'] > 0].sort_values('Quantidade_Nulos', ascending=True)

if not nulos_filtrados.empty:
    bars = ax.barh(nulos_filtrados.index, nulos_filtrados['Quantidade_Nulos'], color='#f28e2b', edgecolor='black', linewidth=0.8)
    for bar in bars:
        xval = bar.get_width()
        ax.text(xval + 500, bar.get_y() + bar.get_height()/2, f"{xval:,}", ha='left', va='center', fontweight='bold', fontsize=9)
    ax.set_title('Contagem de Valores Ausentes (Nulos) por Atributo', fontsize=12, fontweight='bold', pad=15)
    ax.set_xlabel('Quantidade de Valores Nulos', fontsize=10, fontweight='bold')
    ax.grid(axis='x', linestyle='--', alpha=0.5)
else:
    ax.text(0.5, 0.5, "Nenhum valor nulo identificado", ha='center', va='center', fontsize=12)

plt.tight_layout()
plt.show()

df_clean = df_dedup.dropna(subset=['Horário do Alarme', 'TAG']).copy()
print(f"Linhas restantes após remover nulos críticos em 'Horário do Alarme' e 'TAG': {len(df_clean):,}")



## 4. Padronização e Parsing de Campos Temporais

O campo `Tempo de Normalização` é fornecido originalmente como string formatada em `X dias, HH:MM:SS`. Para permitir cálculos estatísticos e filtragem por limiares operacionais, aplicamos uma função de conversão para transformar esse texto em segundos numéricos (`tempo_normalizacao_seg`).

Além disso, convertemos as colunas `Horário do Alarme` e `Horário da Normalizacao` em objetos `datetime` nativos e recalculamos a diferença direta entre as marcas de tempo para validação cruzada.


In [ ]:
def parse_tempo_normalizacao(val):
    if pd.isna(val): return np.nan
    val_str = str(val).strip()
    try:
        if 'dias,' in val_str:
            dias_part, time_part = val_str.split('dias,')
            dias = int(dias_part.strip().split()[0])
        else:
            dias = 0
            time_part = val_str
        h, m, s = map(int, time_part.strip().split(':'))
        return dias * 86400 + h * 3600 + m * 60 + s
    except Exception:
        return np.nan

df_clean['tempo_normalizacao_seg'] = df_clean['Tempo de Normalização'].apply(parse_tempo_normalizacao)

df_clean['dt_alarme'] = pd.to_datetime(df_clean['Horário do Alarme'], errors='coerce')
df_clean['dt_normalizacao'] = pd.to_datetime(df_clean['Horário da Normalizacao'], errors='coerce')

df_clean['duracao_calculada_seg'] = (df_clean['dt_normalizacao'] - df_clean['dt_alarme']).dt.total_seconds()

display(df_clean[['Horário do Alarme', 'Tempo de Normalização', 'tempo_normalizacao_seg', 'duracao_calculada_seg']].head())



## 5. Decomposição Estruturada de TAGs com Tratamento de Nomenclaturas

As TAGs do sistema de portas seguem dois padrões principais de sintaxe:
1. `PSD_EST_EST1_PDM09_TIPO` (ex: `PSD_FAL_FAL2_PDM09_OBS`) -> Estação = `FAL`.
2. `PSD_P_EST1_MDU16_TIPO` (ex: `PSD_P_BUT1_MDU16_STS`) -> A letra `P` indica *Plataforma* e a Estação real é `BUT`.

Abaixo, aplicamos a função de tratamento refinada com Expressão Regular para extrair corretamente as estações reais de todos os registros.


In [ ]:
def extrair_atributos_tag_refinado(tag):
    if pd.isna(tag): 
        return pd.Series([np.nan, np.nan, np.nan, np.nan])
    
    tag_str = str(tag).strip()
    
    match_p = re.search(r'PSD_P_([A-Z]{3})(\d+)_([A-Z0-9]+)_([A-Z0-9]+)', tag_str)
    if match_p:
        estacao = match_p.group(1)
        plataforma = f"{match_p.group(1)}{match_p.group(2)}"
        porta_match = re.search(r'\d+', match_p.group(3))
        porta = int(porta_match.group(0)) if porta_match else np.nan
        tipo = match_p.group(4)
        return pd.Series([estacao, plataforma, porta, tipo])
        
    match_std = re.search(r'PSD_([A-Z]{3})_([A-Z0-9]+)_(?:PDM|MDU)(\d+)_([A-Z0-9]+)', tag_str)
    if match_std:
        estacao = match_std.group(1)
        plataforma = match_std.group(2)
        porta = int(match_std.group(3))
        tipo = match_std.group(4)
        return pd.Series([estacao, plataforma, porta, tipo])
        
    parts = tag_str.split('_')
    if len(parts) >= 3:
        if parts[1] == 'P':
            est = re.sub(r'\d+', '', parts[2])
            plat = parts[2]
        else:
            est = parts[1]
            plat = parts[2]
        porta_match = re.search(r'\d+', parts[3]) if len(parts) > 3 else None
        porta = int(porta_match.group(0)) if porta_match else np.nan
        tipo = parts[-1] if len(parts) > 4 else np.nan
        return pd.Series([est, plat, porta, tipo])
        
    return pd.Series([np.nan, np.nan, np.nan, np.nan])

df_clean[['estacao_extraida', 'plataforma_extraida', 'porta_extraida', 'tipo_evento_extraido']] = df_clean['TAG'].apply(extrair_atributos_tag_refinado)

fig, ax = plt.subplots(figsize=(12, 4.5))
estacao_counts = df_clean['estacao_extraida'].value_counts()

bars = ax.bar(estacao_counts.index, estacao_counts.values, color='#59a14f', edgecolor='black', linewidth=0.8)
for bar in bars:
    yval = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2, yval + 3000, f"{yval:,}", ha='center', va='bottom', fontsize=8, rotation=30)

ax.set_title('Distribuição Corrigida de Ocorrências por Estação Metroviária (Linha 4-Amarela)', fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel('Estação Metroviária (Sigla Real)', fontsize=10, fontweight='bold')
ax.set_ylabel('Quantidade de Ocorrências', fontsize=10, fontweight='bold')
ax.grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()



## 6. Tratamento de Inconsistências Temporais e Outliers Extremamente Longos

Analisamos duas categorias principais de anomalias de qualidade nos dados temporais:
1. **Durações Negativas:** Eventos onde o horário de normalização precede o horário do alarme (inconsistência de relógio do controlador).
2. **Outliers Extremos de Longa Duração (> 30 dias):** Registros em que o alarme permanece em aberto por meses ou anos. Isso ocorre quando uma porta é desativada temporariamente para manutenção pesada sem que o evento no sistema seja encerrado de imediato. Para análises preditivas operacionais, durações acima de 30 dias são consideradas falhas de fechamento de chamado e tratadas/limitadas.


In [ ]:
duracoes_negativas = df_clean[df_clean['tempo_normalizacao_seg'] < 0]
print(f"Registros com tempo de normalização negativo: {len(duracoes_negativas)}")

percentis = df_clean['tempo_normalizacao_seg'].quantile([0.5, 0.75, 0.90, 0.95, 0.99, 0.999])
df_percentis = pd.DataFrame({'Percentil': percentis.index, 'Segundos': percentis.values})
df_percentis['Horas'] = df_percentis['Segundos'] / 3600
df_percentis['Dias'] = df_percentis['Segundos'] / 86400
display(df_percentis)

LIMITE_MAXIMO_DIAS = 30
LIMITE_MAXIMO_SEG = LIMITE_MAXIMO_DIAS * 86400

df_final = df_clean[
    (df_clean['tempo_normalizacao_seg'] >= 0) & 
    (df_clean['tempo_normalizacao_seg'] <= LIMITE_MAXIMO_SEG)
].copy()

removidos_outliers = len(df_clean) - len(df_final)
print(f"Registros removidos por exceder o limite de {LIMITE_MAXIMO_DIAS} dias ({LIMITE_MAXIMO_SEG:,} s): {removidos_outliers:,}")
print(f"Total final de registros limpos e higienizados: {len(df_final):,}")

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))

sns.histplot(df_clean['tempo_normalizacao_seg'] + 1, log_scale=True, ax=ax1, color='#e15759', bins=30)
ax1.set_title('Distribuição Bruta do Tempo de Normalização (Escala Log)', fontsize=11, fontweight='bold')
ax1.set_xlabel('Tempo de Normalização + 1 (Segundos - Escala Log)', fontsize=9)
ax1.set_ylabel('Frequência de Alarmes', fontsize=9)
ax1.axvline(LIMITE_MAXIMO_SEG, color='black', linestyle='--', label='Corte de 30 Dias')
ax1.legend()

sns.histplot(df_final['tempo_normalizacao_seg'] + 1, log_scale=True, ax=ax2, color='#4c78a8', bins=30)
ax2.set_title('Distribuição Sanitizada (< 30 Dias) (Escala Log)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Tempo de Normalização + 1 (Segundos - Escala Log)', fontsize=9)
ax2.set_ylabel('Frequência de Alarmes', fontsize=9)

plt.tight_layout()
plt.show()



## 7. Afunilamento Completo do Pipeline de Limpeza e Exportação

Para demonstrar o impacto de cada etapa do pipeline de tratamento, apresentamos o gráfico de funil com a evolução da volumetria de dados do estado bruto ao conjunto de dados higienizado final.


In [ ]:
etapas = ['1. Dataset Bruto Concatenado', '2. Após Desduplicação', '3. Após Nulos Críticos', '4. Base Limpa Final (<30d)']
volumes = [total_inicial, len(df_dedup), len(df_clean), len(df_final)]

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.barh(etapas[::-1], volumes[::-1], color=['#4c78a8', '#59a14f', '#edc948', '#e15759'], edgecolor='black', linewidth=0.8)

for bar in bars:
    xval = bar.get_width()
    ax.text(xval + 15000, bar.get_y() + bar.get_height()/2, f"{xval:,} ({(xval/total_inicial)*100:.1f}%)", ha='left', va='center', fontweight='bold', fontsize=9)

ax.set_title('Afunilamento de Volumetria ao Longo do Pipeline de Limpeza de Dados', fontsize=12, fontweight='bold', pad=15)
ax.set_xlabel('Quantidade de Registros Mantidos', fontsize=10, fontweight='bold')
ax.set_xlim(0, total_inicial * 1.2)
ax.grid(axis='x', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

output_path = "../../data/Alarmes/alarmes_trata_limpos_consolidado.csv"
df_final.to_csv(output_path, index=False, encoding='utf-8')
print(f"Pipeline concluído com sucesso! Dataset salvo em: {output_path}")

